# 01 — S01 synthetic core workflow and contracts

**Question:** Do feature extraction, integrity, scientific QC, quality scaling, fusion, and matched modelling obey their declared contracts on deterministic synthetic inputs?

**Permitted inputs:** deterministic signals generated inside this notebook. No cohort data, participant identifiers, or E01 result files are read.

**Outputs:** `notebooks/artifacts/01_feature_contracts.json` and `notebooks/artifacts/01_core_workflow.json`.

In [1]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np

from neurofusion_qa.features import eeg_features_33, fisher_connectivity, fmri_features_1440
from neurofusion_qa.integrity import validate_compact_features
from neurofusion_qa.provenance import canonical_json_sha256, sha256_file

START = Path.cwd().resolve()
ROOT = next(
    candidate for candidate in (START, *START.parents)
    if (candidate / 'pyproject.toml').is_file()
    and (candidate / 'src' / 'neurofusion_qa').is_dir()
)
ARTIFACT_DIR = ROOT / 'notebooks' / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 20260918

## Generate deterministic, non-biological fixtures

The fixtures are mathematical test signals. They are not simulations of a disease population and cannot support clinical or population-level claims.

In [2]:
def make_synthetic_inputs(seed: int) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)

    eeg_rate = 250.0
    eeg_time = np.arange(1000, dtype=float) / eeg_rate
    eeg = np.vstack([
        (1.0 + 0.04 * channel) * np.sin(2.0 * np.pi * (3.0 + channel) * eeg_time + 0.2 * channel)
        + 0.35 * np.sin(2.0 * np.pi * 10.0 * eeg_time - 0.1 * channel)
        + rng.normal(0.0, 0.08, size=eeg_time.size)
        for channel in range(8)
    ])

    fmri_rate = 1.25
    fmri_time = np.arange(160, dtype=float) / fmri_rate
    latent = np.vstack([
        np.sin(2.0 * np.pi * 0.03 * fmri_time),
        np.cos(2.0 * np.pi * 0.06 * fmri_time),
        np.sin(2.0 * np.pi * 0.09 * fmri_time + 0.4),
    ])
    loadings = rng.normal(0.0, 0.7, size=(360, 3))
    fmri = loadings @ latent + rng.normal(0.0, 0.35, size=(360, fmri_time.size))
    return eeg, fmri

eeg, fmri = make_synthetic_inputs(SEED)
eeg_repeat, fmri_repeat = make_synthetic_inputs(SEED)
assert np.array_equal(eeg, eeg_repeat)
assert np.array_equal(fmri, fmri_repeat)
assert not np.array_equal(eeg, make_synthetic_inputs(SEED + 1)[0])
{'eeg_shape': eeg.shape, 'fmri_shape': fmri.shape}

{'eeg_shape': (8, 1000), 'fmri_shape': (360, 160)}

## Exercise the public feature interfaces

In [3]:
eeg_features = eeg_features_33(eeg, sampling_rate=250.0)
connectivity = fisher_connectivity(fmri)
fmri_features = fmri_features_1440(fmri, sampling_rate=1.25)

assert eeg_features.shape == (33,)
assert connectivity.shape == (360, 360)
assert fmri_features.shape == (1440,)
assert np.all(np.isfinite(eeg_features))
assert np.all(np.isfinite(connectivity))
assert np.all(np.isfinite(fmri_features))
assert np.allclose(np.diag(connectivity), 0.0)
{'eeg_features': eeg_features.shape, 'connectivity': connectivity.shape, 'fmri_features': fmri_features.shape}

{'eeg_features': (33,), 'connectivity': (360, 360), 'fmri_features': (1440,)}

## Check computational integrity and deliberate failures

These reports describe array structure only. They do not certify scientific QC.

In [4]:
eeg_integrity = validate_compact_features(eeg_features, expected_shape=(33,))
fmri_integrity = validate_compact_features(fmri_features, expected_shape=(1440,))
constant_failure = validate_compact_features(np.ones(33), expected_shape=(33,))
shape_failure = validate_compact_features(eeg_features[:-1], expected_shape=(33,))

assert eeg_integrity.passed
assert fmri_integrity.passed
assert not constant_failure.passed and not constant_failure.positive_variance
assert not shape_failure.passed and shape_failure.observed_shape == (32,)
{
    'eeg_pass': eeg_integrity.to_dict(),
    'fmri_pass': fmri_integrity.to_dict(),
    'constant_failure': constant_failure.to_dict(),
    'shape_failure': shape_failure.to_dict(),
}

{'eeg_pass': {'passed': True,
  'expected_shape': (33,),
  'observed_shape': (33,),
  'all_finite': True,
  'positive_variance': True},
 'fmri_pass': {'passed': True,
  'expected_shape': (1440,),
  'observed_shape': (1440,),
  'all_finite': True,
  'positive_variance': True},
 'constant_failure': {'passed': False,
  'expected_shape': (33,),
  'observed_shape': (33,),
  'all_finite': True,
  'positive_variance': False},
 'shape_failure': {'passed': False,
  'expected_shape': (33,),
  'observed_shape': (32,),
  'all_finite': True,
  'positive_variance': True}}

## Freeze compact evidence

Array hashes make feature ordering changes visible without publishing signal arrays. Hash equality is a regression aid, not proof of scientific correctness.

In [5]:
def array_sha256(values: np.ndarray) -> str:
    canonical = np.ascontiguousarray(values, dtype='<f8')
    digest = hashlib.sha256()
    digest.update(json.dumps(list(canonical.shape), separators=(',', ':')).encode('ascii'))
    digest.update(canonical.tobytes(order='C'))
    return digest.hexdigest()

report = {
    'schema_version': '1.0',
    'notebook_id': '01_S01_Synthetic_Feature_Contracts',
    'boundary': {
        'input_scope': 'deterministic_mathematical_fixtures_only',
        'e01_participant_level_access': False,
        'e01_results_consumed': False,
        'clinical_claim_supported': False,
    },
    'seed': SEED,
    'inputs': {
        'eeg_shape': list(eeg.shape),
        'fmri_shape': list(fmri.shape),
    },
    'features': {
        'eeg': {
            'shape': list(eeg_features.shape),
            'sha256': array_sha256(eeg_features),
            'finite': bool(np.all(np.isfinite(eeg_features))),
            'ordering': '15 band summaries; 3 ratios; 4 spectrum summaries; 8 correlation summaries; 3 Hjorth summaries',
        },
        'fmri': {
            'shape': list(fmri_features.shape),
            'sha256': array_sha256(fmri_features),
            'finite': bool(np.all(np.isfinite(fmri_features))),
            'ordering': '360 strength; 360 ALFF; 360 fALFF; 360 regional SD',
        },
        'connectivity': {
            'shape': list(connectivity.shape),
            'sha256': array_sha256(connectivity),
            'zero_diagonal': bool(np.allclose(np.diag(connectivity), 0.0)),
        },
    },
    'integrity': {
        'eeg_pass': eeg_integrity.to_dict(),
        'fmri_pass': fmri_integrity.to_dict(),
        'constant_failure': constant_failure.to_dict(),
        'shape_failure': shape_failure.to_dict(),
    },
}
report['canonical_content_sha256'] = canonical_json_sha256(report)
artifact_path = ARTIFACT_DIR / '01_feature_contracts.json'
artifact_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n', encoding='utf-8')
reloaded = json.loads(artifact_path.read_text(encoding='utf-8'))
stored_hash = reloaded.pop('canonical_content_sha256')
assert canonical_json_sha256(reloaded) == stored_hash
{'artifact': artifact_path.relative_to(ROOT).as_posix(), 'sha256': sha256_file(artifact_path)}

{'artifact': 'notebooks/artifacts/01_feature_contracts.json',
 'sha256': 'a61e0cc088213e0297969b6f38341d95b06369539e9f5eb76a8d2c1e6458c8fa'}

## Extend the synthetic workflow

The feature examples above exercise signal extraction. The modelling section below uses separate compact mathematical arrays to exercise classification interfaces; it does not claim those arrays came from EEG/fMRI preprocessing. Comparative performance belongs to the frozen protocol in notebook 03.

Reusable operations are defined in `scripts/s01_core_workflow.py`, so the notebook and integration tests execute the same contracts.

In [6]:
import importlib.util
from importlib.metadata import version

helper_path = ROOT / 'scripts' / 's01_core_workflow.py'
spec = importlib.util.spec_from_file_location('s01_core_workflow', helper_path)
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)

## Computational integrity and scientific QC

Vary one measurement at a time while the other six criteria pass. All seven criteria are tested at their declared boundary and either side where applicable. Then show all four combinations of integrity and QC outcomes.

A scientific QC rejection leaves the computational integrity report intact.

In [7]:
qc_evidence = workflow.qc_contracts()
assert len(qc_evidence['threshold_cases']) == 20
assert len(qc_evidence['independent_gates']) == 4
qc_evidence

{'baseline_measurements': {'censored_fraction': 0.1,
  'registration_dice': 0.9,
  'centroid_distance_mm': 2.0,
  'parcels_present': 360,
  'parcels_with_10_voxels': 360,
  'finite_connectome': True,
  'minimum_filtered_regional_sd': 0.1},
 'threshold_cases': [{'criterion': 'censored_fraction',
   'value': 0.19,
   'expected_pass': True,
   'observed_pass': True},
  {'criterion': 'censored_fraction',
   'value': 0.2,
   'expected_pass': True,
   'observed_pass': True},
  {'criterion': 'censored_fraction',
   'value': 0.21,
   'expected_pass': False,
   'observed_pass': False},
  {'criterion': 'registration_dice',
   'value': 0.69,
   'expected_pass': False,
   'observed_pass': False},
  {'criterion': 'registration_dice',
   'value': 0.7,
   'expected_pass': True,
   'observed_pass': True},
  {'criterion': 'registration_dice',
   'value': 0.71,
   'expected_pass': True,
   'observed_pass': True},
  {'criterion': 'centroid_distance_mm',
   'value': 7.9,
   'expected_pass': True,
   'obse

## Training-only quality scaling

Use training percentiles for held-out imputation and clipping. Changing held-out values must leave fitted percentiles unchanged. A deliberately leaky refit shows why the training boundary matters. Availability controls a modality's contribution, and the quality path accepts no labels.

In [8]:
quality_evidence = workflow.quality_contracts()
quality_evidence

{'training_median': [1.5, 20.0],
 'training_p10': [0.30000000000000004, 12.0],
 'training_p90': [2.7, 36.0],
 'scaled_held_out': [[0.0, 0.0], [5.0, -5.0]],
 'availability_scores': [0.5, 0.0],
 'held_out_transform_preserves_fit': True,
 'leaky_refit_changes_percentiles': True,
 'quality_path_uses_labels': False}

## Fusion invariants and failure diagnostics

Exercise 512 generated valid cases, probability endpoints, equal weighting, and utility independence. Invalid probabilities, shapes, weights, directions, availability, and zero or negative quality must produce specific errors.

In [9]:
fusion_evidence = workflow.fusion_contracts(seed=SEED)
assert len(fusion_evidence['failures']) == 9
fusion_evidence

{'generated_valid_cases': 512,
 'weight_normalization': True,
 'probability_bounds': True,
 'endpoint_recovery': True,
 'utility_independent_of_gate': True,
 'failures': [{'case': 'nonfinite_probability',
   'exception': 'ValueError',
   'message': 'p_eeg contains non-finite values'},
  {'case': 'infinite_probability',
   'exception': 'ValueError',
   'message': 'p_eeg contains non-finite values'},
  {'case': 'probability_outside_bounds',
   'exception': 'ValueError',
   'message': 'p_eeg must lie in [0, 1]'},
  {'case': 'probability_shape',
   'exception': 'ValueError',
   'message': 'unimodal probabilities must have the same shape'},
  {'case': 'invalid_weight',
   'exception': 'ValueError',
   'message': 'w_fmri must lie in [0, 1]'},
  {'case': 'negative_quality',
   'exception': 'ValueError',
   'message': 'quality scores must be non-negative'},
  {'case': 'zero_total_quality',
   'exception': 'ValueError',
   'message': 'at least one modality must have positive quality'},
  {'case

## Matched nested modelling

Use 80 synthetic records, two repeats, four shared outer folds, and three inner folds. For each modality, choose C using inner held-out Brier loss, derive the threshold from those training-only predictions, then fit the final pipeline on outer training rows. Quality scaling uses those same outer training rows.

Every record must be held out exactly once per repeat. The report saves split hashes and aggregate shapes rather than individual predictions. Run the entire operation twice to verify reproducibility within this recorded environment.

In [10]:
model_evidence = workflow.matched_model_contracts(seed=1201, repeats=2)
model_replay = workflow.matched_model_contracts(seed=1201, repeats=2)
assert canonical_json_sha256(model_evidence) == canonical_json_sha256(model_replay)
model_evidence['same_seed_replay_matches'] = True
{'records': model_evidence['records'],
 'outer_repeats': model_evidence['outer_repeats'],
 'fold_reports': len(model_evidence['folds']),
 'predictions': model_evidence['predictions'],
 'same_seed_replay_matches': True}

{'records': 80,
 'outer_repeats': 2,
 'fold_reports': 8,
 'predictions': {'eeg': {'repeat_shape': [2, 80],
   'averaged_shape': [80],
   'minimum': 0.01822717725236824,
   'maximum': 0.9955667825001022,
   'rounded_probability_sha256': '5666a4cb36bff11eef4d32aac949132aad9afa4de98a497fc0e7398ac393227e'},
  'fmri': {'repeat_shape': [2, 80],
   'averaged_shape': [80],
   'minimum': 0.014732192552226936,
   'maximum': 0.9669719192475443,
   'rounded_probability_sha256': 'e98a4d8bd7301321987a583d9b70c29b4dd409fbc56d738d172f0fedcce55be5'},
  'equal': {'repeat_shape': [2, 80],
   'averaged_shape': [80],
   'minimum': 0.051563471759539424,
   'maximum': 0.9185032346714906,
   'rounded_probability_sha256': 'c61910a0b3fc10bd31d4fcc3f8cee0e0c15ed02a4a5a453c27195944642699da'},
  'quality': {'repeat_shape': [2, 80],
   'averaged_shape': [80],
   'minimum': 0.05073928152184108,
   'maximum': 0.922532896469531,
   'rounded_probability_sha256': '8690245736ae822dc43552dfd44ae8145f6a8694db541e4f9750e8b0

## Save the complete core evidence record

Record source hashes and dependencies alongside all checks. These fingerprints identify the local source state; they do not imply that an unchanged 0.1.0 release already includes this work. The final release audit will bind evidence to a candidate tag and archive.

In [11]:
core_report = {
    'schema_version': '1.0',
    'notebook_id': '01_S01_Synthetic_Core_Workflow',
    'boundary': report['boundary'],
    'environment': {name: version(name) for name in ('neurofusion-qa', 'numpy', 'scipy', 'scikit-learn')},
    'source_sha256': {
        path.relative_to(ROOT).as_posix(): sha256_file(path)
        for path in [helper_path, *sorted((ROOT / 'src' / 'neurofusion_qa').glob('*.py'))]
    },
    'feature_evidence': {
        'path': 'notebooks/artifacts/01_feature_contracts.json',
        'file_sha256': sha256_file(artifact_path),
    },
    'qc': qc_evidence,
    'quality': quality_evidence,
    'fusion': fusion_evidence,
    'modelling': model_evidence,
}
core_report['canonical_content_sha256'] = canonical_json_sha256(core_report)
core_path = ARTIFACT_DIR / '01_core_workflow.json'
core_path.write_text(json.dumps(core_report, indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')
verified = json.loads(core_path.read_text(encoding='utf-8'))
expected_hash = verified.pop('canonical_content_sha256')
assert canonical_json_sha256(verified) == expected_hash
assert verified['boundary']['e01_results_consumed'] is False
{'artifact': core_path.relative_to(ROOT).as_posix(), 'sha256': sha256_file(core_path)}

{'artifact': 'notebooks/artifacts/01_core_workflow.json',
 'sha256': '5938f9ec95308a5096adc16e5e5a908beaed971dda64e7884461cdefeb02bb60'}

## Evidence-bound figure exports

Regenerate relevant figures from this execution's evidence. Every family has editable SVG/PDF and 1000 dpi Times New Roman PNG exports, a caption, and source hashes. [Full gallery](figures/README.md). Synthetic software evidence only.

In [12]:
import sys
from IPython.display import Markdown, display
if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))
from s01_evidence_figures import generate
figure_records = generate('01')
for item in figure_records:
    display(Markdown(f"### {item['id']}\n\n![{item['id']}](figures/previews/{item['id']}.png)\n\n{item['caption']}"))
{'figure_sets': len(figure_records), 'export_formats': ['png', 'svg', 'pdf']}

### F01_feature_contracts

![F01_feature_contracts](figures/previews/F01_feature_contracts.png)

Deterministic mathematical inputs, identical to notebook 01. Mixed EEG feature units must not be compared as effect sizes; fMRI block distributions are normalized only for display; connectivity is not an anatomical brain map.

### F02_qc_boundaries

![F02_qc_boundaries](figures/previews/F02_qc_boundaries.png)

Seven declared fMRI QC criteria checked at/around their boundaries. Green passes; orange rejects. QC and feature integrity are distinct: neither substitutes for the other. Binary criterion has only two cases; values are in each criterion's native units.

### F03_quality_response

![F03_quality_response](figures/previews/F03_quality_response.png)

Missing quality imputes to the training median; extremes clip to [-5,5]. Fusion response follows q_fMRI/(q_EEG+q_fMRI); both-zero quality is rejected, not drawn. Monotonic response is a contract, not proof of predictive benefit.

{'figure_sets': 3, 'export_formats': ['png', 'svg', 'pdf']}

Exported F01_feature_contracts: Times New Roman, 1000 dpi
Exported F02_qc_boundaries: Times New Roman, 1000 dpi
Exported F03_quality_response: Times New Roman, 1000 dpi
